# 🛡️ Project Panopticon: Intelligent Proctoring AI
## EduGuard AI — Executive Jupyter Notebook

**Objective:** Build a time-series classification pipeline that detects suspicious exam behavior while mathematically prioritizing precision and minimizing false accusations.

### Required project traps addressed
1. **Asynchronous merge:** align video telemetry and event logs with `pandas.merge_asof`.
2. **Connection dropped:** preserve the exam timeline and impute missing telemetry.
3. **Micro-movement noise:** use 10-second rolling features.
4. **False accusation:** use `predict_proba()` and a strict **0.90** decision threshold.

> **Important modeling note:** The supplied `system_events.csv` contains the `is_cheating` label. After a backward `merge_asof`, the latest event label is carried forward until the next event. This follows the project specification, but the resulting label semantics should be validated before any real-world deployment.

## 0. Imports and reproducibility

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    precision_recall_curve,
    average_precision_score,
)

RANDOM_STATE = 42
plt.rcParams["figure.figsize"] = (10, 6)

print("Libraries imported successfully.")

## Phase 1 — Data ingestion and temporal alignment

The video telemetry is sampled every second, while system events occur only when an event happens. Therefore, a normal `pd.merge()` is not appropriate. We use `pd.merge_asof(..., direction="backward")` after converting and sorting timestamps.

In [ ]:
# Locate the supplied CSV files.
# The first two names are convenient local/Colab names; the uploaded names are fallbacks.

video_candidates = [
    "video_telemetry.csv",
    "video_telemetry (1) (1).csv",
]
system_candidates = [
    "system_events.csv",
    "system_events (1) (1).csv",
]

def find_file(candidates):
    for name in candidates:
        if Path(name).exists():
            return Path(name)
    # Also search the current directory for a matching prefix.
    for p in Path(".").glob("*.csv"):
        if any(p.name.startswith(Path(c).stem) for c in candidates):
            return p
    return None

video_path = find_file(video_candidates)
system_path = find_file(system_candidates)

if video_path is None or system_path is None:
    raise FileNotFoundError(
        "Place video_telemetry.csv and system_events.csv in the notebook's working directory."
    )

video_df = pd.read_csv(video_path)
sys_df = pd.read_csv(system_path)

video_df["timestamp"] = pd.to_datetime(video_df["timestamp"])
sys_df["timestamp"] = pd.to_datetime(sys_df["timestamp"])

video_df = video_df.sort_values("timestamp").reset_index(drop=True)
sys_df = sys_df.sort_values("timestamp").reset_index(drop=True)

print("Video telemetry shape:", video_df.shape)
print("System events shape:", sys_df.shape)
print("\nVideo columns:", list(video_df.columns))
print("System-event columns:", list(sys_df.columns))

In [ ]:
# Asynchronous temporal alignment.
# A system event is associated with the most recent event at or before each video timestamp.

merged_df = pd.merge_asof(
    video_df,
    sys_df,
    on="timestamp",
    direction="backward",
)

print("Merged shape:", merged_df.shape)
display(merged_df.head())

print("\nMissing values immediately after merge:")
display(merged_df.isna().sum())

## Phase 2 — Missing data and time-series feature engineering

The video feed contains missing telemetry values. We preserve the timeline rather than deleting missing sensor rows.

- `eye_gaze_angle`: forward-fill the last observed gaze position.
- `audio_db`: interpolate between known measurements.
- Event fields: fill missing values with zero before modeling.
- Rolling windows: calculate sustained behavior over the preceding 10 seconds.

In [ ]:
# Preserve the original missingness counts for the executive report.
missing_before = merged_df[["eye_gaze_angle", "audio_db"]].isna().sum()

# Required imputation strategy.
merged_df["eye_gaze_angle"] = merged_df["eye_gaze_angle"].ffill()
merged_df["audio_db"] = merged_df["audio_db"].interpolate(method="linear")

# No event before the first recorded event means no observed event at that timestamp.
merged_df["tab_switches"] = merged_df["tab_switches"].fillna(0)
merged_df["is_cheating"] = merged_df["is_cheating"].fillna(0)

print("Missing sensor values before imputation:")
display(missing_before.to_frame("missing_count"))

print("\nMissing values after imputation:")
display(merged_df[["eye_gaze_angle", "audio_db", "tab_switches", "is_cheating"]].isna().sum())

In [ ]:
# 10-second rolling features.
# min_periods=10 follows the project instruction to create a complete 10-second window.

merged_df["gaze_rolling_10s"] = (
    merged_df["eye_gaze_angle"]
    .rolling(window=10, min_periods=10)
    .mean()
)

merged_df["audio_rolling_10s"] = (
    merged_df["audio_db"]
    .rolling(window=10, min_periods=10)
    .max()
)

# Remove only the initial rows that cannot have a complete 10-second rolling window.
rows_before_drop = len(merged_df)
merged_df = merged_df.dropna(subset=["gaze_rolling_10s", "audio_rolling_10s"]).reset_index(drop=True)
rows_dropped = rows_before_drop - len(merged_df)

print("Rows removed for incomplete initial rolling windows:", rows_dropped)
print("Feature-engineered dataframe shape:", merged_df.shape)

display(
    merged_df[
        ["timestamp", "eye_gaze_angle", "audio_db", "tab_switches",
         "gaze_rolling_10s", "audio_rolling_10s", "is_cheating"]
    ].head(12)
)

## Phase 3 — Class-balanced model training

The target is `is_cheating`.

The project specifically requires an imbalance-aware classifier. We therefore use a `RandomForestClassifier` with `class_weight="balanced"`.

For reproducibility, the split uses `random_state=42` and preserves the class ratio with stratification.

In [ ]:
features = [
    "eye_gaze_angle",
    "audio_db",
    "tab_switches",
    "gaze_rolling_10s",
    "audio_rolling_10s",
]

X = merged_df[features].copy()
y = merged_df["is_cheating"].astype(int).copy()

print("Class distribution:")
display(y.value_counts().rename_axis("is_cheating").to_frame("count"))

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

In [ ]:
model = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

model.fit(X_train, y_train)

print("Model training complete.")
print("Class-weight strategy:", model.class_weight)

## Phase 4 — Ethical AI tuning: the 90% decision threshold

The default classifier decision boundary is 0.50. For this project, a student should only be flagged when the model assigns at least **90% probability** to the cheating class.

We therefore evaluate both:
- **0.50 threshold** — standard classification
- **0.90 threshold** — strict, precision-first classification

In [ ]:
# Raw probability of the positive class.
y_proba = model.predict_proba(X_test)
cheating_probabilities = y_proba[:, 1]

# Custom decision thresholds.
DEFAULT_THRESHOLD = 0.50
STRICT_THRESHOLD = 0.90

y_pred_default = (cheating_probabilities >= DEFAULT_THRESHOLD).astype(int)
y_pred_strict = (cheating_probabilities >= STRICT_THRESHOLD).astype(int)

def summarize_threshold(name, y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    return {
        "threshold": name,
        "true_negatives": tn,
        "false_positives": fp,
        "false_negatives": fn,
        "true_positives": tp,
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }

results = pd.DataFrame([
    summarize_threshold("0.50", y_test, y_pred_default),
    summarize_threshold("0.90", y_test, y_pred_strict),
])

display(results.round(4))

print("Confusion matrix — 0.50 threshold")
print(confusion_matrix(y_test, y_pred_default))

print("\nConfusion matrix — 0.90 threshold")
print(confusion_matrix(y_test, y_pred_strict))

### Precision–Recall curve

The Precision–Recall curve shows the trade-off between catching suspicious cases and avoiding false accusations as the probability threshold changes.

In [ ]:
precision, recall, thresholds = precision_recall_curve(
    y_test,
    cheating_probabilities
)
average_precision = average_precision_score(
    y_test,
    cheating_probabilities
)

plt.figure()
plt.plot(recall, precision, label=f"Random Forest (AP = {average_precision:.3f})")
plt.axvline(
    x=recall[np.argmin(np.abs(thresholds - STRICT_THRESHOLD))] if len(thresholds) else 0,
    linestyle="--",
    label="Approx. recall at 0.90 threshold",
)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall Curve")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Directly report the metrics at the required 0.90 threshold.
strict_precision = precision_score(y_test, y_pred_strict, zero_division=0)
strict_recall = recall_score(y_test, y_pred_strict, zero_division=0)

print(f"Average precision: {average_precision:.4f}")
print(f"Strict 0.90 precision: {strict_precision:.4f}")
print(f"Strict 0.90 recall: {strict_recall:.4f}")

## Detailed classification report — strict threshold

In [ ]:
print(classification_report(
    y_test,
    y_pred_strict,
    target_names=["Not Cheating", "Cheating"],
    zero_division=0,
))

## Feature importance

This is included for executive interpretability. It shows which engineered inputs contributed most to the Random Forest's decisions on this supplied dataset.

In [ ]:
importance_df = (
    pd.DataFrame({
        "feature": features,
        "importance": model.feature_importances_,
    })
    .sort_values("importance", ascending=False)
)

display(importance_df)

plt.figure()
plt.barh(importance_df["feature"], importance_df["importance"])
plt.xlabel("Random Forest importance")
plt.ylabel("Feature")
plt.title("Feature Importance")
plt.gca().invert_yaxis()
plt.show()

## 🎯 Phase 5 — Executive recommendation

### Recommendation framework

The 0.90 threshold is intentionally conservative. Increasing the threshold should reduce the number of innocent students flagged as suspicious, but it can also increase false negatives.

Therefore, the system should **not automatically punish or make a final academic-integrity decision** based solely on this classifier. A high-probability flag should be treated as a review signal for an authorized human examiner, with appropriate institutional safeguards.

The final recommendation below is generated from the actual evaluation values above.

In [ ]:
default_fp = int(results.loc[results["threshold"] == "0.50", "false_positives"].iloc[0])
strict_fp = int(results.loc[results["threshold"] == "0.90", "false_positives"].iloc[0])
default_recall = float(results.loc[results["threshold"] == "0.50", "recall"].iloc[0])
strict_recall = float(results.loc[results["threshold"] == "0.90", "recall"].iloc[0])
strict_precision = float(results.loc[results["threshold"] == "0.90", "precision"].iloc[0])

print("EXECUTIVE RECOMMENDATION")
print("=" * 70)
print(f"False positives at 0.50 threshold: {default_fp}")
print(f"False positives at 0.90 threshold: {strict_fp}")
print(f"Precision at 0.90 threshold: {strict_precision:.3f}")
print(f"Recall at 0.50 threshold: {default_recall:.3f}")
print(f"Recall at 0.90 threshold: {strict_recall:.3f}")
print()
print(
    "The strict threshold is appropriate as a precision-first REVIEW signal "
    "because it requires at least 90% predicted probability before flagging."
)
print(
    "However, thresholding alone does not establish that cheating occurred. "
    "Human review and institutional due-process safeguards remain necessary."
)